In [1]:
from pathlib import Path
from datetime import datetime
from typing import List
from types import SimpleNamespace
import pickle
import numpy as np
import torch
from pymatgen.core import Structure, Lattice, Element


from chggen.pl_data.dataset import CHGNetDataset
from chggen.pl_modules.model_old import CHGGen
from chggen.common.data_utils import get_scaler_from_data_list, get_scaler
import os

import pytorch_lightning as pl

def mkdir(path: str):
    folder = os.path.exists(path)
    if not folder:
        os.makedirs(path)
    else:
        print("Folder exists")
    return path

/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

device = torch.device('cuda')


with open('./test_models/lattice_scaler_perov', 'rb') as fp:
    lattice_scaler = pickle.load(fp)


chggen = CHGGen.load_from_checkpoint('./test_models/perov_old/epoch=1.ckpt')
chggen.to(device = device)

chggen.lattice_scaler = lattice_scaler




dataset = CHGNetDataset(
    path='./data/perov_5/test_zpc.csv',
    name = 'A_good_name',
    prop_list = ['heat_all'],
)

# lattice_scaler = get_scaler(dataset= dataset)


CHGNet initialized with 412,525 parameters
CHGNet v0.3.0 initialized with 412,525 parameters


/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/overrides.py:110: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  torch.has_cuda,
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/overrides.py:111: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  torch.has_cudnn,
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/overrides.py:117: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  torch.has_mps,
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/overrides.py:118: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  torch.has_mkldnn,
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use 

In [3]:
graph_list = [dataset[ii].crys_graph.to(chggen.device) for ii in range(len(dataset))]

In [4]:
_, _, z_reconst =  chggen.encode(graph_list)

z_reconst = z_reconst[:10]

In [5]:
# chggen.reparameterize(crystal_features, )

In [6]:
torch.diag(dataset[5].lattices)

tensor([4.0965])

In [ ]:

# langevin dynamics
ld_kwargs = SimpleNamespace(n_step_each = 10,
                            step_lr = 1e-3,
                            min_sigma = 0,
                            save_traj = False,
                            disable_bar = False,
                            compute_force = True,
                            beta_c = 0, # property update rate
                            beta_f = 0, # atomic force update rate
                            )


num_structures = len(z_reconst)
# z = torch.rand(num_structures, 64, requires_grad= True, device = device)
results = chggen.langevin_dynamics_guidance(z = z_reconst, 
                                           prop_guidance = torch.tensor(-0.05, device= device), 
                                           # box_lengths = [4.1, 4.1, 4.1],
                                           # box_angles = [90, 90, 90],
                                           # gt_num_atoms = torch.ones(num_structures, device = device, dtype = torch.int64) * 5, # 
                                           # box_lengths = 4.1*1,
                                           # box_angles = 90,
                                           # change_type = True, 
                                           ld_kwargs= ld_kwargs)


/home/zhongpc/chggen/chggen/common/data_utils.py:637: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)
  0%|                                                                                                         | 0/50 [00:00<?, ?it/s]

CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO


  2%|█▉                                                                                               | 1/50 [00:06<05:02,  6.18s/it]

CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO


  4%|███▉                                                                                             | 2/50 [00:10<04:08,  5.18s/it]

...

CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO


 28%|██████████████████████████▉                                                                     | 14/50 [01:06<02:48,  4.69s/it]

CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO


 30%|████████████████████████████▊                                                                   | 15/50 [01:11<02:42,  4.63s/it]

CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO
CuReOsO2
KNO2F
CuPbN2O
IrWN3
IrRuO2F
SiPdPtO2
NiNOF2
KReSO2
CdBN2O
CaNbPdNO


In [ ]:
STOP

In [ ]:
from chgnet.model import StructOptimizer

relaxer = StructOptimizer(model = chggen.encoder.model)


In [ ]:
mkdir('./test_models/reconst_structures/')

In [ ]:

# save the results from langevin dynamics
lengths = results['lengths']
angles= results['angles']
num_atoms = results['num_atoms']
frac_coords = results['frac_coords']
atom_types = results['atom_types']

batch = torch.arange(len(num_atoms), device = device)
batch = batch.repeat_interleave(num_atoms)
print(num_atoms)
for ii in range(len(num_atoms)):
    indices = torch.where(batch == ii)[0]
    # print(ii, indices, )

    crys_graph = dataset[ii].crys_graph
    # print("composition", crys_graph.composition)
    print("num atoms: ", len(indices))

    if len(indices) == 0:
        continue
    
    
    Latt = Lattice.from_parameters(a = lengths.cpu().detach().numpy()[ii,0], 
                                   b = lengths.cpu().detach().numpy()[ii,1], 
                                   c = lengths.cpu().detach().numpy()[ii,2],
                                   alpha= angles.cpu().detach().numpy()[ii, 0], 
                                   beta = angles.cpu().detach().numpy()[ii,1], 
                                   gamma= angles.cpu().detach().numpy()[ii, 2])
                                   
    frac_ = frac_coords[indices]
    type_ = atom_types[indices]
    species_ = [Element.from_Z(ele_Z) for ele_Z in type_]
    
    s_gen = Structure(lattice= Latt , species= species_, coords= frac_.cpu().detach().numpy(),
                      to_unit_cell=False,coords_are_cartesian=False);
    s_gen.sort()
    print("previou compo: ", crys_graph.composition)
    print("reconst compo: ", s_gen.composition)
    s_gen.to(filename= './test_models/reconst_structures/reconst_' + str(ii) + '.cif')
print("Done")

In [ ]:
result = relaxer.relax(s_gen, fmax= 0.5, steps = 100)
print("CHGNet relaxed structure", result["final_structure"])
print("relaxed total energy in eV:", result['trajectory'].energies[-1])

In [ ]:
result['final_structure'].to(filename='./test_models/reconst_structures/chgnet.cif')

In [ ]:
sigma_begin = 10
sigma_end = 0.1
num_noise_level = 50

sigmas = torch.tensor(np.exp(np.linspace(
            np.log(sigma_begin),
            np.log(sigma_end),
            num_noise_level)), dtype=torch.float32)

In [ ]:
sigmas